# Directionality & topology-specificity controls (the Kirschstein mirror test)

Does the upstream-flow gain **require correct flow direction and the real river topology**, or is
it generic spatial correlation? Kirschstein & Sun (2024) diagnosed GNN failure as *directional
insensitivity* (edges maintained/reversed/permuted → same performance). If our static flow feature
is direction-**sensitive**, we exhibit the property whose absence explains the GNN null.

Stock cudalstm, seed 11, **observed** discharge, lag 1 — the ONLY thing that changes is the edge set:

| Condition | aggregates | edges |
|---|---|---|
| L | (none) | — |
| L+upQ (forward) | true upstream parents | existing oracle |
| **L+upQ_reversed** | downstream children | edges reversed |
| **L+upQ_random** | random basins | degree-preserving rewire |

**Pre-reg** (`preregistration_directionality_controls.md`): forward − reversed ≥ +0.015 (direction
matters) AND forward − random ≥ +0.015 (topology matters). Falsify: reversed ≈ forward → gain is
generic correlation, not routing.

**Idempotent — Runtime → T4 GPU → Run all.** Skips any run already complete; only trains the two
new conditions if the forward oracle + baseline are already on Drive.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # blank -> auto-detect
SEED=11
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; print('Found',c); break
    else: raise RuntimeError('set DRIVE_CAMELS_PATH')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydrology_runs'; os.makedirs(DRIVE_RUNS,exist_ok=True)
print('seed', SEED)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(os.path.join(REPO_DIR,'runs','topology_ablation','component0'),exist_ok=True)
print('symlinks ready')

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Build features (forward oracle + reversed + random)

All observed-discharge, lag 1. Idempotent: skips builds whose feature pickle already exists with a
correctly-named ('date') index.

In [ ]:
%cd {REPO_DIR}
import pickle
FEAT='experiments/topology_ablation/features'
def named_ok(p):
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb')); return d[next(iter(d))].index.name=='date'
!python experiments/topology_ablation/generate_topology_attributes.py 2>&1 | tail -1
# forward oracle feature
if named_ok(f'{FEAT}/upstream_q_component0_lag1.p'):
    print('[skip] forward feature present')
else:
    !python experiments/topology_ablation/build_upstream_discharge_feature.py --network component0 --lag-days 1 2>&1 | tail -1
# reversed + random variants
if named_ok(f'{FEAT}/upstream_q_reversed_component0_lag1.p') and named_ok(f'{FEAT}/upstream_q_random_component0_lag1.p'):
    print('[skip] reversed+random features present')
else:
    !python experiments/topology_ablation/build_directionality_variants.py --network component0 --lag-days 1 2>&1 | tail -3

## Cell 8 — Ensure L baseline + forward oracle exist (idempotent)

These are the Δ reference. No-op if already on Drive from prior runs.

In [ ]:
%cd {REPO_DIR}
import glob
B=f'{REPO_DIR}/runs/topology_ablation/component0'
BASIN='topology_analysis/phase1_network_discovery/outputs/component0_basins.txt'
def done(cond): return os.path.isfile(f'{B}/{cond}_component0_seed{SEED}/test/model_epoch030/test_metrics.csv')
# L baseline
if done('L'):
    print('[skip] L present')
else:
    Ldir=f'{B}/L_component0_seed{SEED}'
    !python experiments/topology_ablation/make_configs.py --network component0 --basin-file {BASIN} --seed {SEED} --device cuda:0 --epochs 30
    !python neuralhydrology/nh_run.py train --config-file experiments/topology_ablation/configs/L_component0_seed{SEED}.yaml 2>&1 | tail -2
    ts=sorted(glob.glob(f'{Ldir}_*'))
    if ts: os.rename(ts[-1], Ldir)
    !python neuralhydrology/nh_run.py evaluate --run-dir {Ldir} --epoch 30 2>&1 | tail -1
# forward oracle
if done('L_upQ'):
    print('[skip] L_upQ (forward) present')
else:
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 --feature-file {FEAT}/upstream_q_component0_lag1.p --cond-name L_upQ 2>&1 | tail -2

## Cell 9 — Train the two directionality conditions (idempotent)

In [ ]:
%cd {REPO_DIR}
# reversed edges
if done('L_upQrev'):
    print('[skip] L_upQrev present')
else:
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 --feature-file {FEAT}/upstream_q_reversed_component0_lag1.p --cond-name L_upQrev 2>&1 | tail -2
# random rewire
if done('L_upQrand'):
    print('[skip] L_upQrand present')
else:
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 --feature-file {FEAT}/upstream_q_random_component0_lag1.p --cond-name L_upQrand 2>&1 | tail -2

## Cell 10 — Verdict: forward vs reversed vs random (paired Δ vs L, forward-connected basins)

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np, pickle
from scipy.stats import wilcoxon
B=f'{REPO_DIR}/runs/topology_ablation/component0'
def nse(cond):
    p=f'{B}/{cond}_component0_seed{SEED}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
feat=pickle.load(open(f'{FEAT}/upstream_q_component0_lag1.p','rb'))
conn=[b for b in feat if np.abs(feat[b]['upstream_q'].values).mean()>0]
L=nse('L')
def delta(cond):
    x=nse(cond)
    if x is None or L is None: return None
    c=[b for b in conn if b in x.index and b in L.index]
    return np.array((x.loc[c]-L.loc[c]).values), c
print(f'=== Directionality controls (seed {SEED}, forward-connected n={len(conn)}) ===\n')
res={}
for cond,label in [('L_upQ','forward (true upstream)'),('L_upQrev','reversed (downstream)'),('L_upQrand','random rewire')]:
    d=delta(cond)
    if d is None: print(f'{label:<28} (missing)'); continue
    dd,c=d; res[cond]=np.median(dd)
    p=wilcoxon(dd,alternative='greater').pvalue if len(dd)>=6 and np.any(dd!=0) else float('nan')
    print(f'{label:<28} median Δ={np.median(dd):+.4f}  frac>0={100*(dd>0).mean():.0f}%  p={p:.1e}')
if all(k in res for k in ['L_upQ','L_upQrev','L_upQrand']):
    gd=res['L_upQ']-res['L_upQrev']; gt=res['L_upQ']-res['L_upQrand']
    print(f'\ndirectional gap (forward − reversed) = {gd:+.4f}   pre-reg >= +0.015 -> {gd>=0.015}')
    print(f'topology gap    (forward − random)   = {gt:+.4f}   pre-reg >= +0.015 -> {gt>=0.015}')
    verdict = (gd>=0.015 and gt>=0.015)
    print(f'\nVERDICT: {"PASS" if verdict else "CHECK"} — '+('the gain requires correct flow direction AND the real river topology; '
          'direction-sensitive where GNNs were not (Kirschstein mirror).' if verdict else
          'gap(s) below +0.015 — read carefully; if reversed≈forward the routing claim needs revision (see pre-reg).'))

## Cell 11 — Persistence check (did the new runs reach Drive?)

In [ ]:
%cd {REPO_DIR}
print('=== persistence check ===')
for cond in ['L_upQrev','L_upQrand']:
    dp=f'{DRIVE_RUNS}/topology_ablation/component0/{cond}_component0_seed{SEED}/test/model_epoch030/test_metrics.csv'
    print(f'{cond:<12} in Drive: {os.path.isfile(dp)}')
print('\nIf both True, the runs are safe. Copy the two run folders back into the repo to file the'
      '\nresult locally (build_paper_table.py / an analyze_directionality.py can then read them).')

## Done

Runs persist to Drive (`neural_hydrology_runs/topology_ablation/component0/L_upQrev...`, `L_upQrand...`).
Report the Cell 10 verdict block + Cell 11 persistence back for filing.